# 02 — Comparaison des Embeddings
**RQ1** : Quel modèle d'embedding maximise le Contextual Recall ?

Teste 4 modèles : bge-m3, e5-large, jina-embeddings-v3, qwen3-embedding.

In [ ]:
import sys, os, pandas as pd, numpy as np
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../src"))

from config import load_config
from retriever import retrieve_documents
from embeddings import create_embeddings
from evaluation_judge import create_judge
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualRecallMetric

from notebooks.lib.reporter import load_benchmark, to_csv, to_json
from notebooks.lib.plotter import boxplot, barplot
from experiments.registry import ExperimentLog

benchmark = load_benchmark()

EMBEDDING_MODELS = [
    "BAAI/bge-m3",
    "intfloat/multilingual-e5-large",
    "jinaai/jina-embeddings-v3",
]

judge = create_judge(load_config().evaluation)

In [ ]:
log = ExperimentLog(name="embedding_comparison")
log.set_params(embeddings=EMBEDDING_MODELS)

recall_metric = ContextualRecallMetric(threshold=0.75, model=judge, include_reason=True)
recall_metric.required_multiple = False

for idx, row in benchmark.iterrows():
    q_id = row['ID']
    question = row['Question']
    expected = row['Ground_Truth']

    for emb in EMBEDDING_MODELS:
        cfg = load_config()
        cfg.embedding.model = emb

        try:
            docs, scores = retrieve_documents(question, cfg.retrieval)
        except Exception as e:
            print(f"[ERR] {q_id} / {emb} : {e}")
            continue

        if not docs:
            log.record(ID=q_id, embedding=emb, num_chunks=0, Contextual_Recall=0.0)
            continue

        contexts = [d.page_content for d in docs]
        tc = LLMTestCase(
            input=question,
            actual_output="",
            expected_output=expected,
            retrieval_context=contexts,
        )

        try:
            recall_metric.measure(tc)
            recall = round(recall_metric.score, 4)
        except:
            recall = 0.0

        log.record(ID=q_id, embedding=emb, num_chunks=len(docs),
                   Contextual_Recall=recall, niveau=row['Niveau_Complexite'])

print("Expérience terminée.")

In [ ]:
# Résultats agrégés
df = pd.DataFrame(log.results)
print("=== Score moyen par embedding ===")
avg = df.groupby('embedding')['Contextual_Recall'].agg(['mean', 'std', 'count']).round(4)
print(avg)

# Boxplot
boxplot(df, x='embedding', y='Contextual_Recall',
        title="Contextual Recall par modèle d'embedding",
        filename='embedding_recall_comparison.png')

# Sauvegarde
log.save_all()
log.append_to_global_log()
print("Résultats sauvegardés dans outputs/")

## Analyse
- Comparer les moyennes et écarts-types
- Identifier l'embedding le plus performant pour chaque niveau de complexité
- Conclusion pour le mémoire : recommandation du meilleur embedding